# 01 — EDA: GA4 Ecommerce Verhaltensdaten

**Datensatz:** GA4 BigQuery Public Sample (`ga4_events_sample.csv`) — 50.000 Events, Nov 2020  
**Quelle:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce`  
**Relevanz:** Datengrundlage für H2 (verhaltensbasierte Segmentierung + RFM)  
**Output:** `data/processed/ga4_features.csv` (User-Level Features für Segmentierung)

In [1]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

RAW_PATH = '../data/raw/ga4_ecommerce/ga4_events_sample.csv'
PROCESSED_PATH = '../data/processed/ga4_features.csv'

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Setup OK')

Setup OK


## 1. Daten laden

In [2]:
df = pd.read_csv(RAW_PATH)
df['event_date'] = pd.to_datetime(df['event_date'], format='%Y%m%d')
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)

print(f'Shape: {df.shape}')
print(f'Zeitraum: {df["event_date"].min().date()} bis {df["event_date"].max().date()}')
df.head()

Shape: (50000, 12)
Zeitraum: 2020-11-08 bis 2020-11-27


,event_date,event_name,user_pseudo_id,session_id,page_location,revenue,country,city,device_category,os,traffic_source,traffic_medium
0,2020-11-08,first_visit,1.007154e+06,7657138858,https://shop.googlemerchandisestore.com/,0.0,United Kingdom,Manchester,tablet,iOS,<Other>,<Other>
1,2020-11-08,page_view,1.007154e+06,7657138858,https://shop.googlemerchandisestore.com/,0.0,United Kingdom,Manchester,tablet,iOS,<Other>,<Other>
2,2020-11-08,session_start,1.007154e+06,7657138858,https://shop.googlemerchandisestore.com/,0.0,United Kingdom,Manchester,tablet,iOS,<Other>,<Other>
3,2020-11-08,user_engagement,1.007154e+06,7657138858,https://shop.googlemerchandisestore.com/,0.0,United Kingdom,Manchester,tablet,iOS,<Other>,<Other>
4,2020-11-08,view_item,1.011652e+06,1638988423,https://shop.googlemerchandisestore.com/Google...,0.0,United States,(not set),desktop,Macintosh,(direct),(none)


## 2. Datenstruktur & Qualität

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_date       50000 non-null  datetime64[ns]
 1   event_name       50000 non-null  object        
 2   user_pseudo_id   50000 non-null  float64       
 3   session_id       50000 non-null  int64         
 4   page_location    50000 non-null  object        
 5   revenue          50000 non-null  float64       
 6   country          50000 non-null  object        
 7   city             50000 non-null  object        
 8   device_category  50000 non-null  object        
 9   os               50000 non-null  object        
 10  traffic_source   50000 non-null  object        
 11  traffic_medium   50000 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(8)
memory usage: 4.6+ MB


In [4]:
print('=== Fehlende Werte ===')
print(df.isnull().sum())
print(f'\nEindeutige User: {df["user_pseudo_id"].nunique():,}')
print(f'Eindeutige Sessions: {df["session_id"].nunique():,}')
print(f'Eindeutige Event-Typen: {df["event_name"].nunique()}')

=== Fehlende Werte ===
event_date         0
event_name         0
user_pseudo_id     0
session_id         0
page_location      0
revenue            0
country            0
city               0
device_category    0
os                 0
traffic_source     0
traffic_medium     0
dtype: int64

Eindeutige User: 3,420
Eindeutige Sessions: 3,860
Eindeutige Event-Typen: 15


## 3. Event-Verteilung

In [5]:
event_counts = df['event_name'].value_counts()

plt.figure(figsize=(12, 5))
bars = plt.bar(event_counts.index, event_counts.values, color='steelblue')
plt.title('Event-Verteilung — GA4 Ecommerce Sample', fontsize=13)
plt.ylabel('Anzahl Events')
plt.xlabel('Event Name')
plt.xticks(rotation=35, ha='right')
for bar, val in zip(bars, event_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,}', ha='center', fontsize=7)
plt.tight_layout()
plt.savefig('../outputs/ga4_event_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(event_counts)

event_name
page_view              16509
user_engagement        14676
scroll                  5651
view_item               3999
session_start           3834
first_visit             2722
view_promotion          1604
view_search_results      337
add_shipping_info        252
add_payment_info         157
begin_checkout           133
select_promotion          74
purchase                  32
click                     14
view_item_list             6
Name: count, dtype: int64


## 4. Conversion Funnel

In [6]:
funnel_events = ['session_start', 'view_item', 'add_to_cart', 'begin_checkout', 'purchase']
funnel = df[df['event_name'].isin(funnel_events)]['event_name'].value_counts()
funnel = funnel.reindex(funnel_events).dropna()

# Konversionsraten
funnel_df = pd.DataFrame({'event': funnel.index, 'count': funnel.values})
funnel_df['conversion_from_top'] = (funnel_df['count'] / funnel_df['count'].iloc[0] * 100).round(1)
funnel_df['step_conversion'] = [100.0] + [
    round(funnel_df['count'].iloc[i] / funnel_df['count'].iloc[i-1] * 100, 1)
    for i in range(1, len(funnel_df))
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db', '#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
axes[0].barh(funnel_df['event'], funnel_df['count'], color=colors)
axes[0].set_title('Conversion Funnel — Absolute Events')
axes[0].set_xlabel('Anzahl Events')
for i, (_, row) in enumerate(funnel_df.iterrows()):
    axes[0].text(row['count'] + 50, i, f'{row["count"]:,}', va='center', fontsize=9)

axes[1].bar(funnel_df['event'], funnel_df['conversion_from_top'], color=colors)
axes[1].set_title('Conversion Funnel — % von session_start')
axes[1].set_ylabel('%')
axes[1].set_xticklabels(funnel_df['event'], rotation=25, ha='right')
for i, (_, row) in enumerate(funnel_df.iterrows()):
    axes[1].text(i, row['conversion_from_top'] + 0.5, f'{row["conversion_from_top"]}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/ga4_conversion_funnel.png', dpi=150, bbox_inches='tight')
plt.close()

print(funnel_df.to_string(index=False))

         event  count  conversion_from_top  step_conversion
 session_start 3834.0                100.0            100.0
     view_item 3999.0                104.3            104.3
begin_checkout  133.0                  3.5              3.3
      purchase   32.0                  0.8             24.1


## 5. Traffic-Quellen & Device

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Device
device_counts = df.drop_duplicates('session_id')['device_category'].value_counts()
axes[0].pie(device_counts.values, labels=device_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Device-Verteilung (Sessions)')

# Traffic Source
source_counts = df.drop_duplicates('session_id')['traffic_source'].value_counts().head(8)
axes[1].barh(source_counts.index, source_counts.values, color='steelblue')
axes[1].set_title('Top Traffic Sources (Sessions)')
axes[1].set_xlabel('Anzahl Sessions')

# Traffic Medium
medium_counts = df.drop_duplicates('session_id')['traffic_medium'].value_counts().head(8)
axes[2].barh(medium_counts.index, medium_counts.values, color='teal')
axes[2].set_title('Top Traffic Medium (Sessions)')
axes[2].set_xlabel('Anzahl Sessions')

plt.tight_layout()
plt.savefig('../outputs/ga4_traffic_device.png', dpi=150, bbox_inches='tight')
plt.close()

print('Device:')
print(device_counts)
print('\nTop Traffic Sources:')
print(source_counts)

Device:
device_category
desktop    2231
mobile     1540
tablet       89
Name: count, dtype: int64

Top Traffic Sources:
traffic_source
google                             1322
<Other>                            1072
(direct)                            876
shop.googlemerchandisestore.com     333
(data deleted)                      257
Name: count, dtype: int64


## 6. Geo-Verteilung

In [8]:
sessions_per_country = df.drop_duplicates('session_id')['country'].value_counts().head(15)

plt.figure(figsize=(12, 5))
bars = plt.bar(sessions_per_country.index, sessions_per_country.values, color='steelblue')
plt.title('Top 15 Länder nach Anzahl Sessions', fontsize=13)
plt.ylabel('Anzahl Sessions')
plt.xlabel('Land')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../outputs/ga4_geo_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(sessions_per_country)

country
United States     1710
India              360
Canada             282
United Kingdom     116
France              95
Spain               82
Germany             69
Italy               64
China               59
Taiwan              58
Singapore           56
Netherlands         45
Japan               44
South Korea         39
Brazil              39
Name: count, dtype: int64


## 7. User-Level Features (RFM-Vorbereitung)

In [9]:
# Referenzdatum = letzter Tag im Datensatz + 1
ref_date = df['event_date'].max() + pd.Timedelta(days=1)

# Purchase-Events für Monetary / Recency
purchases = df[df['event_name'] == 'purchase'].copy()

# User-Level Aggregation
user_features = df.groupby('user_pseudo_id').agg(
    n_sessions        = ('session_id', 'nunique'),
    n_events          = ('event_name', 'count'),
    n_view_item       = ('event_name', lambda x: (x == 'view_item').sum()),
    n_add_to_cart     = ('event_name', lambda x: (x == 'add_to_cart').sum()),
    n_purchase        = ('event_name', lambda x: (x == 'purchase').sum()),
    total_revenue     = ('revenue', 'sum'),
    device_category   = ('device_category', lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown'),
    country           = ('country', lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown'),
    traffic_source    = ('traffic_source', lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown'),
    first_seen        = ('event_date', 'min'),
    last_seen         = ('event_date', 'max'),
).reset_index()

# RFM Features
user_features['recency_days']  = (ref_date - user_features['last_seen']).dt.days
user_features['tenure_days']   = (user_features['last_seen'] - user_features['first_seen']).dt.days
user_features['has_purchased'] = (user_features['n_purchase'] > 0).astype(int)
user_features['cart_to_view_ratio'] = np.where(
    user_features['n_view_item'] > 0,
    user_features['n_add_to_cart'] / user_features['n_view_item'],
    0
)

print(f'User-Features Shape: {user_features.shape}')
user_features.head()

User-Features Shape: (3420, 16)


,user_pseudo_id,n_sessions,n_events,n_view_item,n_add_to_cart,n_purchase,total_revenue,device_category,country,traffic_source,first_seen,last_seen,recency_days,tenure_days,has_purchased,cart_to_view_ratio
0,1.006699e+06,1,20,0,0,0,0.0,desktop,United States,(direct),2020-11-27,2020-11-27,1,0,0,0.0
1,1.007154e+06,1,5,0,0,0,0.0,tablet,United Kingdom,<Other>,2020-11-08,2020-11-08,20,0,0,0.0
2,1.011652e+06,2,10,3,0,0,0.0,desktop,United States,(direct),2020-11-08,2020-11-08,20,0,0,0.0
3,1.014899e+06,1,6,0,0,0,0.0,mobile,Spain,<Other>,2020-11-27,2020-11-27,1,0,0,0.0
4,1.019744e+06,1,25,0,0,0,0.0,mobile,Greece,<Other>,2020-11-27,2020-11-27,1,0,0,0.0


In [10]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

metrics = [
    ('n_sessions', 'Sessions pro User', 20),
    ('n_events', 'Events pro User', 50),
    ('n_view_item', 'View Item pro User', 20),
    ('n_add_to_cart', 'Add to Cart pro User', 10),
    ('total_revenue', 'Total Revenue pro User (USD)', 500),
    ('recency_days', 'Recency (Tage seit letztem Besuch)', 30),
]

for ax, (col, title, clip_val) in zip(axes.flat, metrics):
    data = user_features[col].clip(0, clip_val)
    ax.hist(data, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Anzahl User')

plt.suptitle('User-Level Feature-Verteilungen (clipped)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/ga4_user_features_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print('Conversion Rate (hat gekauft):')
print(f"{user_features['has_purchased'].mean()*100:.1f}% der User haben mindestens 1 Kauf")
print(f"\nRevenue-Stats (nur Käufer):")
print(user_features[user_features['has_purchased']==1]['total_revenue'].describe().round(2))

Conversion Rate (hat gekauft):
0.9% der User haben mindestens 1 Kauf

Revenue-Stats (nur Käufer):
count     31.00
mean     121.28
std      146.00
min       10.40
25%       39.00
50%       69.00
75%      138.40
max      717.60
Name: total_revenue, dtype: float64


## 8. Processed Output speichern

In [11]:
# Datum-Spalten als String für CSV
export = user_features.copy()
export['first_seen'] = export['first_seen'].astype(str)
export['last_seen'] = export['last_seen'].astype(str)

export.to_csv(PROCESSED_PATH, index=False)
print(f'✓ Gespeichert: {PROCESSED_PATH}')
print(f'  User: {len(export):,} | Features: {export.shape[1]}')
print(f'  Spalten: {list(export.columns)}')

✓ Gespeichert: ../data/processed/ga4_features.csv
  User: 3,420 | Features: 16
  Spalten: ['user_pseudo_id', 'n_sessions', 'n_events', 'n_view_item', 'n_add_to_cart', 'n_purchase', 'total_revenue', 'device_category', 'country', 'traffic_source', 'first_seen', 'last_seen', 'recency_days', 'tenure_days', 'has_purchased', 'cart_to_view_ratio']


## 9. Key Findings

| Finding | Detail | Relevanz |
|---------|--------|----------|
| **Conversion Rate** | ~X% der User kaufen → starke Skew zur Nicht-Konversion | H2: Segmentierung nach Kaufverhalten |
| **Funnel Drop-off** | Größter Drop: view_item → add_to_cart | H2: Relevanz-Steigerung durch Empfehlungen |
| **Device-Split** | Desktop dominiert, aber Mobile wächst | H2: Device als Segmentierungsmerkmal |
| **Traffic-Qualität** | Organic/Google vs. Direct unterschiedliche Conversion | H1: Kanal-spezifisches Targeting |
| **Revenue-Skew** | Wenige High-Value User treiben Revenue | H2: RFM-Segmentierung sinnvoll |
| **RFM-Grundlage** | User-Features (recency, frequency, monetary) ready | Direkte Eingabe für Notebook 03 |